In [0]:
spark

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

Creating widgets 

In [0]:
dbutils.widgets.text("catalog", "consumer_goods", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

Using widgets value

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = f'/Volumes/consumer_goods/files/input_data_files/child_company/{data_source}/*.csv'

print(catalog)
print(data_source)
print(base_path)


consumer_goods
customers
/Volumes/consumer_goods/files/input_data_files/child_company/customers/*.csv


Creating df along with extra columns like timestamp and filename

In [0]:
df = (spark.read.format("csv")
        .option("header", "True")
        .option("inferSchema", "True")
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*","_metadata.file_name")
)


In [0]:
df.show(10)

+-----------+--------------------+---------+--------------------+-------------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|
+-----------+--------------------+---------+--------------------+-------------+
|     789201|      FitFuel Market|Bengaluru|2026-03-03 07:10:...|customers.csv|
|     789202|      FitFuel Market|Hyderabad|2026-03-03 07:10:...|customers.csv|
|     789203|      FitFuel Market|New Delhi|2026-03-03 07:10:...|customers.csv|
|     789301|Athlete's Choice ...|Bengaluru|2026-03-03 07:10:...|customers.csv|
|     789303|Athlete's Choice ...|New Delhi|2026-03-03 07:10:...|customers.csv|
|     789101|     Endurance Foods|Bengalore|2026-03-03 07:10:...|customers.csv|
|     789102|     Endurance Foods|Hyderabad|2026-03-03 07:10:...|customers.csv|
|     789103|     Endurance Foods|New Delhi|2026-03-03 07:10:...|customers.csv|
|     789121| HydroBoost Nutri...|Hyderabad|2026-03-03 07:10:...|customers.csv|
|     789122|HydroBoost Nutrition|New De

Creating a bronze table with raw data

In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").saveAsTable(f"{catalog}.bronze.{data_source}")

Transformations or cleanup

In [0]:
display(df)

customer_id,customer_name,city,read_timestamp,file_name
789201,FitFuel Market,Bengaluru,2026-03-03T07:10:32.352Z,customers.csv
789202,FitFuel Market,Hyderabad,2026-03-03T07:10:32.352Z,customers.csv
789203,FitFuel Market,New Delhi,2026-03-03T07:10:32.352Z,customers.csv
789301,Athlete's Choice Store,Bengaluru,2026-03-03T07:10:32.352Z,customers.csv
789303,Athlete's Choice Store,New Delhi,2026-03-03T07:10:32.352Z,customers.csv
789101,Endurance Foods,Bengalore,2026-03-03T07:10:32.352Z,customers.csv
789102,Endurance Foods,Hyderabad,2026-03-03T07:10:32.352Z,customers.csv
789103,Endurance Foods,New Delhi,2026-03-03T07:10:32.352Z,customers.csv
789121,HydroBoost Nutrition,Hyderabad,2026-03-03T07:10:32.352Z,customers.csv
789122,HydroBoost Nutrition,New Delhi,2026-03-03T07:10:32.352Z,customers.csv


Finding duplicates based on customer_id

In [0]:
df_duplicates = df.groupBy("customer_id").count().filter(F.col("count") >1)
display(df_duplicates)

customer_id,count
789321,2
789503,2
789522,2
789603,2


Remove duplicates

In [0]:
df_no_duplicates = df.dropDuplicates(["customer_id"])
print("Before :",df.count())
print("After :",df_no_duplicates.count())

Before : 39
After : 35


trim spaces in customer_name

In [0]:
df_no_duplicates=df_no_duplicates.withColumn("customer_name", F.trim(F.col("customer_name")))
display(df_no_duplicates)

customer_id,customer_name,city,read_timestamp,file_name
789101,Endurance Foods,Bengalore,2026-03-03T07:10:42.218Z,customers.csv
789102,Endurance Foods,Hyderabad,2026-03-03T07:10:42.218Z,customers.csv
789103,Endurance Foods,New Delhi,2026-03-03T07:10:42.218Z,customers.csv
789121,HydroBoost Nutrition,Hyderabad,2026-03-03T07:10:42.218Z,customers.csv
789122,HydroBoost Nutrition,New Delhi,2026-03-03T07:10:42.218Z,customers.csv
789201,FitFuel Market,Bengaluru,2026-03-03T07:10:42.218Z,customers.csv
789202,FitFuel Market,Hyderabad,2026-03-03T07:10:42.218Z,customers.csv
789203,FitFuel Market,New Delhi,2026-03-03T07:10:42.218Z,customers.csv
789220,MacroBite Superfoods,Bengaluru,2026-03-03T07:10:42.218Z,customers.csv
789221,MacroBite superfoods,Hyderabadd,2026-03-03T07:10:42.218Z,customers.csv


City names to be fixed

In [0]:
df_no_duplicates.select("city").distinct().show()


+----------+
|      city|
+----------+
| New Delhi|
| Hyderabad|
| Bengaluru|
|      NULL|
|  Hyderbad|
|  NewDelhi|
|Bengaluruu|
| Bengalore|
|  NewDheli|
|Hyderabadd|
| NewDelhee|
+----------+



Correcting city names

In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed = ["Bengaluru", "Hyderabad", "New Delhi"]
df_no_duplicates=(
    df_no_duplicates
    .replace(city_mapping, subset=["city"])
    .withColumn("city", 
                F.when(F.col("city").isNull(),None)
                .when(F.col("city").isin(allowed), F.col("city"))
                .otherwise(None))
)
display(df_no_duplicates)

customer_id,customer_name,city,read_timestamp,file_name
789101,Endurance Foods,Bengaluru,2026-03-03T07:10:44.226Z,customers.csv
789102,Endurance Foods,Hyderabad,2026-03-03T07:10:44.226Z,customers.csv
789103,Endurance Foods,New Delhi,2026-03-03T07:10:44.226Z,customers.csv
789121,HydroBoost Nutrition,Hyderabad,2026-03-03T07:10:44.226Z,customers.csv
789122,HydroBoost Nutrition,New Delhi,2026-03-03T07:10:44.226Z,customers.csv
789201,FitFuel Market,Bengaluru,2026-03-03T07:10:44.226Z,customers.csv
789202,FitFuel Market,Hyderabad,2026-03-03T07:10:44.226Z,customers.csv
789203,FitFuel Market,New Delhi,2026-03-03T07:10:44.226Z,customers.csv
789220,MacroBite Superfoods,Bengaluru,2026-03-03T07:10:44.226Z,customers.csv
789221,MacroBite superfoods,Hyderabad,2026-03-03T07:10:44.226Z,customers.csv


Fixing customer names

In [0]:
df_no_duplicates.select("customer_name").distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|EliteAthlete Nutr...|
|      PowerSnack hub|
|       Recovery Lane|
|      GamePlan Foods|
|      FitFuel Market|
|Athlete's Choice ...|
|    ZenAthlete foods|
|HydroBoost Nutrition|
|    ZenAthlete Foods|
|   champion's Choice|
|Peak Performance ...|
|Peak performance ...|
| PrimeFuel Nutrition|
|   Champion's choice|
|     Endurance Foods|
|      StaminaX Store|
|   SprintX Nutrition|
|   SprintX nutrition|
|      PowerSnack Hub|
|MacroBite Superfoods|
+--------------------+
only showing top 20 rows


using initcap to fix

In [0]:
df_no_duplicates = (
    df_no_duplicates.withColumn("customer_name",
    F.when(F.col("customer_name").isNull(), None)
    .otherwise(F.initcap(F.col("customer_name"))))
)
display(df_no_duplicates)

customer_id,customer_name,city,read_timestamp,file_name
789101,Endurance Foods,Bengaluru,2026-03-03T07:10:45.961Z,customers.csv
789102,Endurance Foods,Hyderabad,2026-03-03T07:10:45.961Z,customers.csv
789103,Endurance Foods,New Delhi,2026-03-03T07:10:45.961Z,customers.csv
789121,Hydroboost Nutrition,Hyderabad,2026-03-03T07:10:45.961Z,customers.csv
789122,Hydroboost Nutrition,New Delhi,2026-03-03T07:10:45.961Z,customers.csv
789201,Fitfuel Market,Bengaluru,2026-03-03T07:10:45.961Z,customers.csv
789202,Fitfuel Market,Hyderabad,2026-03-03T07:10:45.961Z,customers.csv
789203,Fitfuel Market,New Delhi,2026-03-03T07:10:45.961Z,customers.csv
789220,Macrobite Superfoods,Bengaluru,2026-03-03T07:10:45.961Z,customers.csv
789221,Macrobite Superfoods,Hyderabad,2026-03-03T07:10:45.961Z,customers.csv


checking for null city names

In [0]:
df_no_duplicates.filter(F.col("city").isNull()).show()

+-----------+-------------------+----+--------------------+-------------+
|customer_id|      customer_name|city|      read_timestamp|    file_name|
+-----------+-------------------+----+--------------------+-------------+
|     789420|   Zenathlete Foods|NULL|2026-03-03 07:10:...|customers.csv|
|     789403|  Sprintx Nutrition|NULL|2026-03-03 07:10:...|customers.csv|
|     789521|Primefuel Nutrition|NULL|2026-03-03 07:10:...|customers.csv|
|     789603|      Recovery Lane|NULL|2026-03-03 07:10:...|customers.csv|
+-----------+-------------------+----+--------------------+-------------+



checking based on the other values of custome_name

In [0]:
null_customer_name=['Zenathlete Foods','Sprintx Nutrition','Primefuel Nutrition','Recovery Lane']
df_no_duplicates.filter(F.col("customer_name").isin(null_customer_name)).show()

+-----------+-------------------+---------+--------------------+-------------+
|customer_id|      customer_name|     city|      read_timestamp|    file_name|
+-----------+-------------------+---------+--------------------+-------------+
|     789601|      Recovery Lane|Bengaluru|2026-03-03 07:10:...|customers.csv|
|     789420|   Zenathlete Foods|     NULL|2026-03-03 07:10:...|customers.csv|
|     789421|   Zenathlete Foods|Hyderabad|2026-03-03 07:10:...|customers.csv|
|     789520|Primefuel Nutrition|Bengaluru|2026-03-03 07:10:...|customers.csv|
|     789403|  Sprintx Nutrition|     NULL|2026-03-03 07:10:...|customers.csv|
|     789521|Primefuel Nutrition|     NULL|2026-03-03 07:10:...|customers.csv|
|     789401|  Sprintx Nutrition|Bengaluru|2026-03-03 07:10:...|customers.csv|
|     789402|  Sprintx Nutrition|Hyderabad|2026-03-03 07:10:...|customers.csv|
|     789522|Primefuel Nutrition|New Delhi|2026-03-03 07:10:...|customers.csv|
|     789603|      Recovery Lane|     NULL|2026-03-0

Based on data assigning a value to city

In [0]:
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}
df_fix = spark.createDataFrame(
    [(x, y) for x, y in customer_city_fix.items() ],
    ["customer_id","fixed_city"]
)
df_fix.show()

+-----------+----------+
|customer_id|fixed_city|
+-----------+----------+
|     789403| New Delhi|
|     789420| Bengaluru|
|     789521| Hyderabad|
|     789603| Hyderabad|
+-----------+----------+



In [0]:
df_no_duplicates =(
    df_no_duplicates.join(df_fix,"customer_id","left")
    .withColumn("city", F.coalesce("city","fixed_city"))
    .drop("fixed_city")
)
display(df_no_duplicates)


customer_id,customer_name,city,read_timestamp,file_name
789101,Endurance Foods,Bengaluru,2026-03-03T07:10:49.983Z,customers.csv
789102,Endurance Foods,Hyderabad,2026-03-03T07:10:49.983Z,customers.csv
789103,Endurance Foods,New Delhi,2026-03-03T07:10:49.983Z,customers.csv
789121,Hydroboost Nutrition,Hyderabad,2026-03-03T07:10:49.983Z,customers.csv
789122,Hydroboost Nutrition,New Delhi,2026-03-03T07:10:49.983Z,customers.csv
789201,Fitfuel Market,Bengaluru,2026-03-03T07:10:49.983Z,customers.csv
789202,Fitfuel Market,Hyderabad,2026-03-03T07:10:49.983Z,customers.csv
789203,Fitfuel Market,New Delhi,2026-03-03T07:10:49.983Z,customers.csv
789220,Macrobite Superfoods,Bengaluru,2026-03-03T07:10:49.983Z,customers.csv
789221,Macrobite Superfoods,Hyderabad,2026-03-03T07:10:49.983Z,customers.csv


Casting customer_id to string

In [0]:
df_silver = df_no_duplicates.withColumn("customer_id",F.col("customer_id").cast("string"))
df_silver.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)



Building data similar to Gold table

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)
df_silver.show(10)

+-----------+--------------------+---------+--------------------+-------------+--------------------+------+----------+-----------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|            customer|market|  platform|    channel|
+-----------+--------------------+---------+--------------------+-------------+--------------------+------+----------+-----------+
|     789622|Eliteathlete Nutr...|New Delhi|2026-03-03 07:10:...|customers.csv|Eliteathlete Nutr...| India|Sports Bar|Acquisition|
|     789321|      Powersnack Hub|Hyderabad|2026-03-03 07:10:...|customers.csv|Powersnack Hub-Hy...| India|Sports Bar|Acquisition|
|     789601|       Recovery Lane|Bengaluru|2026-03-03 07:10:...|customers.csv|Recovery Lane-Ben...| India|Sports Bar|Acquisition|
|     789720|      Gameplan Foods|Bengaluru|2026-03-03 07:10:...|customers.csv|Gameplan Foods-Be...| India|Sports Bar|Acquisition|
|     789201|      Fitfuel Market|Bengaluru|2026-03-03 07:10:...|customers.csv|Fitf

In [0]:
df_silver.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.silver.{data_source}")

In [0]:
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.gold.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, f"{catalog}.gold.dim_customers")
df_child_customers = spark.table(f"{catalog}.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

Merging sportsbar customer data to actual customers table

In [0]:
delta_table.alias("target").merge(
    source = df_child_customers.alias("source"),
    condition = "target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
describe extended consumer_goods.gold.dim_customers

col_name,data_type,comment
customer_code,string,null
customer,string,null
market,string,null
platform,string,null
channel,string,null
,,
# Delta Statistics Columns,,
Column Names,"market, customer, platform, customer_code, channel",
Column Selection Method,first-32,
,,


In [0]:
%sql
DESCRIBE HISTORY consumer_goods.gold.dim_customers

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-03-03T07:21:55.000Z,71320613334569,yvishnu224@gmail.com,MERGE,"Map(predicate -> [""(customer_code#15079 = customer_code#15057)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3430562047940079),61c4b1e8-57c9-4b59-81c6-7a39d62086b8,0303-070937-cpcqpwvy-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2204, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2994, materializeSourceTimeMs -> 4, numTargetRowsInserted -> 35, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1692, numTargetRowsUpdated -> 0, numOutputRows -> 35, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 35, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1141)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-02T11:41:53.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),e22cf9de-d132-4e16-bae5-1137f657d0ef,0302-113816-lp5wn68f-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1967, numDeletionVectorsRemoved -> 0, numOutputRows -> 18, numOutputBytes -> 1967)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-02-28T06:18:09.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),bf7c37c6-7670-4042-82a4-43f3bb8ea865,0228-052951-nxc2v7k3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 18, numOutputBytes -> 1967)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
